# Finding checks for `001754-knierim-2000-03` (dandiset 001754)

One section per finding about the data that attempt `001754-knierim-2000-03` recorded, in the record's order. Each
section quotes the finding, then runs one cell, tagged `finding:<n>`, that prints the evidence for it
from the files of dandiset 001754 version 0.260728.1352 and states, with `check()`, the conditions the
finding asserts. A verifier can read a finding and its cell side by side, run one cell alone after the
setup cell, and see at the end which conditions hold.

These checks were written after the attempt was verified, to help people verify its findings. They are
not part of the attempt: its outcomes, findings and verified notebook (`reproduce_knierim_2000.ipynb`)
are unchanged. Most of the code is taken from that notebook's data check, reorganised so that each
finding has its own cell. Where a finding compares the data with the paper, the paper's numbers are
written in the cell with where the paper gives them.

Statements that the cells cannot check from the files (what a NASA record holds, what the paper means)
are said to be not checked in the cell.

In [ ]:
import re
import warnings
from collections import Counter

import h5py
import numpy as np
import pandas as pd
import remfile
import requests
from scipy import stats

import neurolab as nl

pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

sessions = nl.load_all()
by_label = {nl.label(s): s for s in sessions}
print(f"loaded {len(sessions)} files of dandiset {nl.DANDISET_ID} version {nl.DANDISET_VERSION}:", sorted(by_label))

results = []

def check(finding, condition, holds, detail=""):
    """Record and print one condition a finding asserts, and whether the data bear it out."""
    holds = bool(holds)
    results.append({"finding": finding, "condition": condition, "holds": holds, "detail": detail})
    print(f"[{'holds' if holds else 'DOES NOT HOLD'}] finding {finding}: {condition}" + (f" ({detail})" if detail else ""))

def read_attr(path_url, key):
    with h5py.File(remfile.File(path_url[1]), "r") as f:
        return nl._text(f[key][()]) if key in f else None

_links = None
def unit_map_links():
    """For each flight unit and task epoch with 50 or more spikes at a position: the correlation of the unit's
    spike-count map with rate x occupancy of every distinct deposited map of its tetrode and epoch, the best
    match, and whether the table's stated link (cell_number = cluster id, same tetrode) exists and names it.
    Shared by findings 5 and 6; computed once."""
    global _links
    if _links is not None:
        return _links
    rows = []
    for s in sessions:
        if "xy" not in s:
            continue
        for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
            if typ == "BL":
                continue
            cand = s["maps"][(s["maps"].epoch == name) & ~s["maps"].duplicate]
            for _, u in s["units"].iterrows():
                ok, r, c = nl.spike_positions(s, u.spikes[(u.spikes >= a) & (u.spikes <= b)])
                if ok.sum() < 50:
                    continue
                counts = nl.count_map(r, c)
                cc = cand[cand.tetrode == u.tetrode]
                rs = {i: np.corrcoef((np.clip(s["rate_maps"][i], 0, None) * nl.clean_occupancy(s["occupancy_maps"][i])).ravel(), counts.ravel())[0, 1]
                      for i in cc.index}
                rs = {i: v for i, v in rs.items() if np.isfinite(v)}
                stated = list(cc.index[cc.cell_number == u.cluster])
                best = max(rs, key=rs.get) if rs else None
                row = {"session": nl.label(s), "epoch": name, "tetrode": u.tetrode, "cluster": int(u.cluster),
                       "best_r": rs[best] if best is not None else np.nan, "best_cell_number": int(cc.cell_number[best]) if best is not None else None,
                       "stated_link_exists": bool(stated), "stated_link_is_best": best in stated}
                if best is not None:
                    rate_i, occ_i = s["rate_maps"][best], nl.clean_occupancy(s["occupancy_maps"][best])
                    positive = (rate_i > 0) & (occ_i > 0)
                    row["positive_bins_without_spikes"] = float((positive & (counts == 0)).sum() / max(positive.sum(), 1))
                rows.append(row)
    _links = pd.DataFrame(rows)
    return _links

## Finding 1: metadata_error

> Still present (partly fixed in the unpublished draft). The Rat1 and Rat2 files of each flight day hold one tracker's trace over the whole session (98.6 to 98.8% of positions at shared timestamps within 2 pixels), each presented as that rat's position, and nothing names the animal tracked. With real against time-shifted spikes, the trace behaves as Rat2's position in Rat2's flight day 9 ES window (12 of 28 Rat2 units, 0 of 36 Rat1 units), and in Rat2's flight day 4 ES window neither rat's units are selective (0 of 17 and 1 of 17), so whose position it is there is not established; there the two files hold the same trace (4 of 707 shared valid positions more than 2 pixels apart), so neither file's copy settles it.

In [ ]:
# One tracker's trace in both rats' files of a flight day. At timestamps both files hold, with a valid
# position in both, the share of positions within 2 pixels; then the time span each file covers.
for day in ["FD4", "FD9"]:
    a, b = by_label[f"Rat1 {day}"], by_label[f"Rat2 {day}"]
    common, ia, ib = np.intersect1d(a["t"], b["t"], return_indices=True)
    both = a["valid"][ia] & b["valid"][ib]
    d = np.hypot(*(a["xy"][ia][both] - b["xy"][ib][both]).T)
    share = (d < 2).mean()
    print(f"{day}: {len(common)} shared timestamps, {int(both.sum())} valid in both, within 2 px {share:.4f}; "
          f"Rat1 spans {a['t'][0]:.0f}-{a['t'][-1]:.0f} s, Rat2 {b['t'][0]:.0f}-{b['t'][-1]:.0f} s")
    check(1, f"{day}: 98.6 to 98.8% of shared positions within 2 px", 0.985 <= share <= 0.989, f"{share:.4f}")

# What each file says the position is (nothing names the tracked animal).
for lab in ["Rat1 FD4", "Rat2 FD4", "Rat1 FD9", "Rat2 FD9"]:
    print(f"{lab}: position description {by_label[lab]['xy_description']!r}")
# "Animal position" says nothing of which animal: the check is for a rat's name or number.
names_animal = any(re.search(r"\brat ?[123]\b|\brat\s+(one|two|three)\b", by_label[l]["xy_description"], re.I) for l in ["Rat1 FD4", "Rat2 FD4", "Rat1 FD9", "Rat2 FD9"])
check(1, "no position description names which rat is tracked", not names_animal)

# In Rat2's FD4 ES window the two files hold the same trace.
r1, r2 = by_label["Rat1 FD4"], by_label["Rat2 FD4"]
a_, b_ = r2["epochs"].set_index("name").loc["ES1", ["start", "stop"]]
m1, m2 = (r1["t"] >= a_) & (r1["t"] <= b_), (r2["t"] >= a_) & (r2["t"] <= b_)
common, i1, i2 = np.intersect1d(r1["t"][m1], r2["t"][m2], return_indices=True)
both = r1["valid"][m1][i1] & r2["valid"][m2][i2]
far = int((np.hypot(*(r1["xy"][m1][i1][both] - r2["xy"][m2][i2][both]).T) > 2).sum())
print(f"Rat2 FD4 ES window {a_:.0f}-{b_:.0f} s: {int(both.sum())} shared valid positions, {far} more than 2 px apart")
check(1, "Rat2 FD4 ES window: 4 of 707 shared valid positions more than 2 px apart", (far, int(both.sum())) == (4, 707), f"{far} of {int(both.sum())}")

# Whose position: units carry more spatial information with real spike times than with every one of 10
# time shifts (+-30..150 s, circular within the window; real spikes subsampled to the shifted count).
SHIFTS = [s_ * k for k in (30, 60, 90, 120, 150) for s_ in (1, -1)]
rng = np.random.default_rng(1754)

def selective_units(units_s, trace_s, a, b, occ):
    occ = nl.clean_occupancy(occ)
    def info(rows, cols):
        counts = nl.count_map(rows, cols); counts[occ == 0] = 0
        return nl.skaggs_info(np.where(occ > 0, counts / np.where(occ > 0, occ, 1), 0.0), occ) if counts.sum() else np.nan
    def lookup(x):
        t = trace_s["t"]; i = np.clip(np.searchsorted(t, x), 1, len(t) - 1)
        j = np.where(np.abs(t[i - 1] - x) <= np.abs(t[i] - x), i - 1, i)
        keep = (np.abs(t[j] - x) <= 0.25) & trace_s["valid"][j]
        return keep, *nl.bin_xy(trace_s["xy"][j[keep]])
    n = above = 0
    for _, u in units_s["units"].iterrows():
        sp = u.spikes[(u.spikes >= a) & (u.spikes <= b)]
        keep, r, c = lookup(sp)
        if keep.sum() < 50:
            continue
        diffs = []
        for sh in SHIFTS:
            k2, r2_, c2 = lookup(a + (sp - a + sh) % (b - a))
            pick = rng.choice(len(r), min(int(k2.sum()), len(r)), replace=False)
            diffs.append((info(r[pick], c[pick]) if len(pick) else np.nan) - (info(r2_, c2) if k2.sum() else np.nan))
        diffs = np.array(diffs)
        n += 1; above += bool(np.all(diffs[np.isfinite(diffs)] > 0))
    return above, n

for day, expect in [("FD9", {"Rat2": (12, 28), "Rat1": (0, 36)}), ("FD4", {"Rat2": (0, 17), "Rat1": (1, 17)})]:
    w = by_label[f"Rat2 {day}"]
    a_, b_ = w["epochs"].set_index("name").loc["ES1", ["start", "stop"]]
    for rat in ["Rat2", "Rat1"]:
        u = by_label[f"{rat} {day}"]
        above, n = selective_units(u, u, a_, b_, nl.epoch_occupancy(w, "ES1"))
        p = stats.binomtest(above, n, 1 / 11, alternative="greater").pvalue
        print(f"Rat2's {day} ES window, {rat}'s units: {above} of {n} above every shift (binomial p vs 1/11: {p:.2g})")
        # The real spikes are subsampled at random, so a count can differ from the record's by a unit or two.
        check(1, f"Rat2 {day} ES window: {expect[rat][0]} of {expect[rat][1]} of {rat}'s units selective (within 2)",
              n == expect[rat][1] and abs(above - expect[rat][0]) <= 2, f"{above} of {n}, p {p:.2g}")

## Finding 2: data_missing

> Still present. Position is given only at spike times, so occupancy time cannot be computed from it, and the preflight files have no position at all.

In [ ]:
# Position only at spike times: every position timestamp is a spike time of the file's own units.
for s in sessions:
    if "xy" not in s:
        print(f"{nl.label(s)}: no position")
        continue
    own = np.unique(np.concatenate(list(s["units"]["spikes"])))
    at_spikes = np.isin(s["t"], own).mean()
    print(f"{nl.label(s)}: {len(s['t'])} position samples, at the file's own spike times: {at_spikes:.4f}; "
          f"description {s['xy_description']!r}")
    check(2, f"{nl.label(s)}: every position sample is at a spike time", at_spikes == 1.0, f"{at_spikes:.4f}")
pre = [nl.label(s) for s in sessions if s["day"] == "preflight"]
check(2, "no preflight file has position", all("xy" not in by_label[l] for l in pre), ", ".join(pre))

## Finding 3: metadata_error

> Still present in this version (the draft's description drops it). The dandiset description says the preflight recordings were made on a flat rectangular track, but the preflight files label their epochs ES (Escher staircase) and MC (Magic Carpet) and hold ES and MC rate maps. The paper states the rectangular track for rat 2 only.

In [ ]:
# The dandiset description against the preflight files' epochs and maps.
meta = requests.get(f"https://api.dandiarchive.org/api/dandisets/001754/versions/{nl.DANDISET_VERSION}/", timeout=60).json()
desc = meta["description"]
for sentence in re.split(r"(?<=\.) ", desc):
    if re.search(r"rectangular|preflight|pre-flight", sentence, re.I):
        print("Description:", sentence)
check(3, "the dandiset description puts the preflight recordings on a rectangular track",
      re.search(r"pre-?flight[^.]*rectangular|rectangular[^.]*pre-?flight", desc, re.I))
for s in sessions:
    if s["day"] == "preflight":
        types = sorted(set(s["epochs"].type)); maps = dict(Counter(s["maps"].type))
        print(f"{nl.label(s)}: epoch types {types}, rate maps by type {maps}")
        check(3, f"{nl.label(s)} labels its epochs ES and MC and holds ES and MC maps",
              types == ["ES", "MC"] and set(maps) == {"ES", "MC"})
print("Not checked here: that the paper states the rectangular track for rat 2 only (Fig. 2b legend).")

## Finding 4: metadata_error

> New. The files' experiment description says baseline sessions were on a rectangular track, interleaved with task sessions; the paper says the baseline sessions were in the sleeping pouch, and the rectangular track was rat 2's preflight task. The preflight files have no baseline epoch.

In [ ]:
# The files' experiment description of baseline sessions, and the preflight files' epochs.
for path_url in nl.asset_urls():
    text = read_attr(path_url, "general/experiment_description") or ""
    hits = [x for x in re.split(r"(?<=\.) ", text) if re.search(r"baseline|rectangular", x, re.I)]
    print(path_url[0].split("/")[1][:31], "|", " ".join(hits))
texts = [read_attr(p, "general/experiment_description") or "" for p in nl.asset_urls()]
check(4, "the experiment description puts baseline sessions on a rectangular track",
      all(re.search(r"baseline[^.]*rectangular|rectangular[^.]*baseline", t, re.I) for t in texts))
for s in sessions:
    if s["day"] == "preflight":
        check(4, f"{nl.label(s)} has no baseline epoch", "BL" not in set(s["epochs"].type), str(sorted(set(s["epochs"].name))))
print("Not checked here: that the paper places baseline sessions in the sleeping pouch (EEG paragraph).")

## Finding 5: identifier_missing

> Still present (fixed in the draft). The rate-map table says cell_number matches the cluster id on the same tetrode, but the stated link exists for 98 of 211 flight unit-epochs and names the best-matching map for 84. Every one of Rat3's 42 flight day 4 ES maps has cell_number -1 ("not a per-cell map") though their names are per-cell short names, so Rat3's session 2 and session 3 maps cannot be paired by cell; tying each to the unit it best matches is not one to one in session 3. Elsewhere 2 to 10 active maps per file and epoch have cell_number -1, and in flight these match one unit about as well as the numbered maps (median correlation 0.555 against 0.595), so they appear to be per-cell maps without their cell number.

In [ ]:
# Does cell_number link a rate map to its unit, as the rate-map table says?
links = unit_map_links()
n, exists, best = len(links), int(links.stated_link_exists.sum()), int(links.stated_link_is_best.sum())
print(f"Flight unit-epochs with >= 50 spikes at a position: {n}; the stated link exists for {exists}, names the best-matching map for {best}")
print(links.groupby(["session", "epoch"]).agg(n=("best_r", "size"), stated_exists=("stated_link_exists", "sum"),
      stated_is_best=("stated_link_is_best", "sum")).to_string())
check(5, "the stated link exists for 98 of 211 and names the best map for 84", (n, exists, best) == (211, 98, 84), f"{exists} and {best} of {n}")

s3 = by_label["Rat3 FD4"]
es = s3["maps"][s3["maps"].type == "ES"]
print(f"\nRat3 FD4 ES maps: {len(es)}, with cell_number -1: {int((es.cell_number == -1).sum())}; names:", sorted(set(es.source_file)))
check(5, "all 42 of Rat3's FD4 ES maps have cell_number -1", len(es) == 42 and (es.cell_number == -1).all(), f"{int((es.cell_number == -1).sum())} of {len(es)}")

# Tying each Rat3 FD4 session 3 (ES2) map to the unit it best matches: is it one to one?
C = nl.map_unit_matches(s3, "ES2")
by_map = C.loc[C.groupby("map_row").r.idxmax()]
twice = by_map.groupby(["tetrode", "cluster"]).size()
twice = twice[twice > 1]
print("Rat3 FD4 session 3: units tied to two or more maps:", twice.to_dict())
check(5, "tying Rat3's session 3 maps to their best units is not one to one", len(twice) > 0, f"{len(twice)} units tied to 2+ maps")

# Elsewhere: active maps with cell_number -1 per file and epoch, and how well they match one unit in flight.
ma = nl.ma_table(sessions)
neg = ma[ma.active & (ma.cell_number == -1) & ~((ma.rat == "Rat3") & (ma.day == "FD4") & (ma.type == "ES"))]
per = neg.groupby(["rat", "day", "epoch"]).size()
print("\nActive maps with cell_number -1, other files and epochs:", per.to_dict())
check(5, "elsewhere 2 to 10 active maps per file and epoch have cell_number -1", per.min() >= 2 and per.max() <= 10, f"{per.min()} to {per.max()}")
# Each active map of the other flight files: its best correlation with a single unit of its tetrode.
rows = []
for s in sessions:
    if "xy" not in s or nl.label(s) == "Rat3 FD4":
        continue
    for name in s["epochs"].name[s["epochs"].type != "BL"]:
        for i, g in nl.map_unit_matches(s, name).groupby("map_row"):
            m = ma[(ma.rat == s["subject"]) & (ma.day == s["day"]) & (ma.row == i)].iloc[0]
            if m.active:
                rows.append({"cell_number_-1": m.cell_number == -1, "best_single_unit_r": g.r.max()})
per_map = pd.DataFrame(rows).groupby("cell_number_-1").best_single_unit_r.agg(["size", "median"])
print("Active maps of the other flight files, best single-unit correlation:"); print(per_map.round(3).to_string())
med_neg, med_num = per_map.loc[True, "median"], per_map.loc[False, "median"]
check(5, "active maps with cell_number -1 match one unit about as well as numbered maps (median 0.555 against 0.595)",
      abs(med_neg - 0.555) < 0.005 and abs(med_num - 0.595) < 0.005, f"{med_neg:.3f} against {med_num:.3f}")

## Finding 6: metadata_insufficient

> Still present (partly fixed in the draft). The rate maps are smoothed (91% of the positive-rate bins of a well-matched map hold no spike of the unit), and neither that nor the kernel is stated.

In [ ]:
# Smoothing: in maps that match their unit well (r > 0.5), the share of positive-rate bins where the unit
# fired no spike. An unsmoothed rate map is zero wherever the unit did not fire.
links = unit_map_links()
good = links[links.best_r > 0.5]
med = good.positive_bins_without_spikes.median()
print(f"Best-matching maps with r > 0.5: {len(good)}; positive-rate bins without a spike: median {med:.3f}, "
      f"quartiles {good.positive_bins_without_spikes.quantile(0.25):.3f} to {good.positive_bins_without_spikes.quantile(0.75):.3f}")
check(6, "91% of the positive-rate bins of a well-matched map hold no spike", abs(med - 0.91) < 0.01, f"{med:.3f}")
rm_desc = []
for path_url in nl.asset_urls():
    with h5py.File(remfile.File(path_url[1]), "r") as f:
        g = f["processing/ecephys/rate_maps"]
        rm_desc.append(" ".join([nl._text(g.attrs.get("description", ""))] + [nl._text(g[c].attrs.get("description", "")) for c in g if isinstance(g[c], h5py.Dataset)]))
print("Rate-map table and column descriptions (first file):", rm_desc[0])
check(6, "no description mentions smoothing or a kernel", not any(re.search(r"smooth|kernel|gaussian", d, re.I) for d in rm_desc))

## Finding 7: metadata_insufficient

> Still present (fixed in the draft). Rate maps use -999 for unvisited bins, which the column description (rate in Hz) does not state, and the occupancy counts the lost position (0, 255) in bin [63, 0], up to 19% of a flight occupancy map.

In [ ]:
# -999 for unvisited bins, and the lost position counted in bin [63, 0].
r_all = np.concatenate([s["rate_maps"].ravel() for s in sessions])
o_all = np.concatenate([s["occupancy_maps"].ravel() for s in sessions])
same = ((r_all == nl.UNVISITED) == (o_all == 0)).all()
print(f"Bins at -999: {(r_all == nl.UNVISITED).mean():.3f}; -999 exactly where occupancy is 0: {same}")
check(7, "rate maps are -999 exactly where occupancy is 0", same)
with h5py.File(remfile.File(nl.asset_urls()[0][1]), "r") as f:
    col = nl._text(f["processing/ecephys/rate_maps/rate_map"].attrs.get("description", ""))
print("rate_map column description:", repr(col))
check(7, "the rate_map column description does not mention -999", "999" not in col)
fractions = {}
for s in sessions:
    for _, _, idx in nl.occupancy_groups(s):
        occ = s["occupancy_maps"][idx[0]]
        fractions[f"{nl.label(s)} {s['maps'].epoch[idx[0]]}"] = occ[nl.SENTINEL_BIN] / occ.sum()
print("Occupancy in bin [63, 0] as a share of the map:", {k: round(v, 3) for k, v in fractions.items()})
flight_max = max(v for k, v in fractions.items() if "preflight" not in k)
check(7, "bin [63, 0] holds up to 19% of a flight occupancy map", abs(flight_max - 0.19) < 0.01, f"max {flight_max:.3f}")

## Finding 8: other

> Still present (fixed in the draft). Six groups of identical rate maps are stored in three files, some under different tetrodes (Rat1 FD4 TT1 and TT11; Rat2 FD4 TT13, TT19 and TT22; Rat2 FD9 TT13 and TT14).

In [ ]:
# Identical rate maps within a file.
groups_found = []
for s in sessions:
    groups = {}
    for i, r in enumerate(s["rate_maps"]):
        groups.setdefault(r.tobytes(), []).append(i)
    for rows_ in groups.values():
        if len(rows_) > 1:
            tets = [s["maps"].tetrode[i] for i in rows_]
            groups_found.append((nl.label(s), tets))
            print(f"{nl.label(s)}: {s['maps'].type[rows_[0]]} copies on " + ", ".join(f"{t}/{s['maps'].source_file[i]}" for t, i in zip(tets, rows_)))
check(8, "six groups of identical maps in three files", len(groups_found) == 6 and len({g[0] for g in groups_found}) == 3,
      f"{len(groups_found)} groups in {len({g[0] for g in groups_found})} files")
tet_sets = {(f, tuple(sorted(set(t)))) for f, t in groups_found if len(set(t)) > 1}
print("Groups whose copies are on different tetrodes:", sorted(tet_sets))
for expect in [("Rat1 FD4", ("TT1", "TT11")), ("Rat2 FD4", ("TT13", "TT19", "TT22")), ("Rat2 FD9", ("TT13", "TT14"))]:
    check(8, f"{expect[0]}: identical maps under {', '.join(expect[1])}", any(f == expect[0] and set(expect[1]) <= set(t) for f, t in tet_sets))

## Finding 9: data_missing

> Still present (partly fixed in the draft's description). The dandiset description reports normal theta and sharp waves and ripples in the hippocampal EEG, but no file holds EEG, and no file name in the NASA record OSD-968 suggests it.

In [ ]:
# EEG: the description reports theta and ripples; does any file hold a voltage series?
meta = requests.get(f"https://api.dandiarchive.org/api/dandisets/001754/versions/{nl.DANDISET_VERSION}/", timeout=60).json()
for sentence in re.split(r"(?<=\.) ", meta["description"]):
    if re.search(r"theta|ripple|EEG", sentence, re.I):
        print("Description:", sentence)
check(9, "the dandiset description reports theta and ripples", re.search(r"theta", meta["description"], re.I) and re.search(r"ripple", meta["description"], re.I))
types = Counter()
for path_url in nl.asset_urls():
    with h5py.File(remfile.File(path_url[1]), "r") as f:
        f.visititems(lambda name, obj: types.update([nl._text(obj.attrs.get("neurodata_type", ""))]) if isinstance(obj, h5py.Group) else None)
print("Neurodata types of groups, over all files:", dict(types))
check(9, "no file holds ElectricalSeries, LFP or any voltage series", not types.keys() & {"ElectricalSeries", "LFP", "FilteredEphys", "DecompositionSeries"})
print("Not checked here: the file names in the NASA record OSD-968.")

## Finding 10: metadata_insufficient

> Still present (fixed in the draft). Spike times are resolved to 1 ms or finer below 1000 s, 10 ms to 10000 s and 0.1 s beyond, with zero intervals that follow, and this is not stated; Rat1's flight day 4 file has two overlapping baseline epochs (2525 to 3390 s and 2720 to 3390 s), and 1 to 14 spikes per file lie outside every epoch.

In [ ]:
# Spike-time resolution by time range, zero intervals within a unit, overlapping epochs, spikes outside epochs.
rows = []
for s in sessions:
    allsp = np.concatenate(list(s["units"].spikes))
    for lo, hi in [(0, 1000), (1000, 10000), (10000, np.inf)]:
        distinct = np.unique(allsp[(allsp >= lo) & (allsp < hi)])
        if len(distinct) < 2:
            continue
        n = z = 0
        for sp in s["units"].spikes:
            w = sp[(sp >= lo) & (sp < hi)]; n += max(len(w) - 1, 0); z += int((np.diff(w) == 0).sum())
        rows.append({"session": nl.label(s), "range_s": f"{lo}-{hi}", "smallest_step_s": round(float(np.diff(distinct).min()), 4),
                     "zero_interval_share": round(z / max(n, 1), 3)})
res = pd.DataFrame(rows)
print(res.to_string(index=False))
step = res.groupby("range_s").smallest_step_s.max()
check(10, "spike times resolved to 1 ms or finer below 1000 s", step.get("0-1000", 1) <= 0.001, str(step.get("0-1000")))
check(10, "to 10 ms from 1000 to 10000 s", abs(step.get("1000-10000", 0) - 0.01) < 1e-6, str(step.get("1000-10000")))
check(10, "to 0.1 s beyond 10000 s", abs(step.get("10000-inf", 0) - 0.1) < 1e-6, str(step.get("10000-inf")))
ep = by_label["Rat1 FD4"]["epochs"]
bl = ep[ep.type == "BL"][["name", "start", "stop"]]
print("\nRat1 FD4 baseline epochs:"); print(bl.to_string(index=False))
iv = sorted(zip(bl.start, bl.stop))
check(10, "Rat1 FD4 has two overlapping baseline epochs (2525-3390 s and 2720-3390 s)",
      len(iv) >= 2 and any(b2 < e1 for (b1, e1), (b2, e2) in zip(iv, iv[1:])), str([(round(a), round(b)) for a, b in iv]))
outside = {nl.label(s): int(sum((~np.any([(sp >= a) & (sp <= b) for a, b in zip(s["epochs"].start, s["epochs"].stop)], axis=0)).sum()
                                for sp in s["units"].spikes)) for s in sessions}
print("Spikes outside every epoch, per file:", outside)
check(10, "1 to 14 spikes per file lie outside every epoch", min(outside.values()) >= 1 and max(outside.values()) <= 14,
      f"{min(outside.values())} to {max(outside.values())}")
with h5py.File(remfile.File(nl.asset_urls()[0][1]), "r") as f:
    st = f["units/spike_times"]
    note = {k: (nl._text(v) if isinstance(v, (bytes, str)) else v) for k, v in st.attrs.items() if k in ("description", "resolution")}
print("units/spike_times attributes:", note)
res_attr = note.get("resolution")
check(10, "the files do not state the spike-time resolution",
      (res_attr is None or float(res_attr) < 0) and not re.search(r"resolution|rounded|precision", str(note.get("description", "")), re.I), str(note))

## Finding 11: metadata_insufficient

> Still present (partly fixed in the draft). The age is "/" for every subject, and the position's pixel scale and camera view are not described.

In [ ]:
# Subject age, and what the position says about its scale and view.
ages = {}
for path_url in nl.asset_urls():
    ages[path_url[0].split("/")[1][:31]] = read_attr(path_url, "general/subject/age")
print("Ages:", ages)
check(11, 'the age is "/" for every subject', set(ages.values()) == {"/"}, str(set(ages.values())))
flight = [s for s in sessions if "xy" in s]
descs = {s["xy_description"] for s in flight}
print("Position descriptions:", descs, "; units:", {s["xy_unit"] for s in flight})
check(11, "no position description gives a pixel scale or camera view",
      not any(re.search(r"cm|mm|scale|per pixel|camera|overhead|view", d, re.I) for d in descs))

## Finding 12: metadata_insufficient

> Still present. Rat2's preflight units are on tetrodes TT0 to TT10 and its flight units on TT12 to TT22, with no tetrode in common, while Rat1 and Rat3 share 4 to 6 tetrodes between preflight and flight; nothing in the files confirms the preflight files are assigned to the right rats.

In [ ]:
# Tetrodes shared between each rat's preflight and flight files.
for rat in ["Rat1", "Rat2", "Rat3"]:
    pre = set(by_label[f"{rat} preflight"]["units"].tetrode)
    fl = set().union(*(set(by_label[f"{rat} {d}"]["units"].tetrode) for d in ["FD4", "FD9"] if f"{rat} {d}" in by_label))
    shared = {d: len(pre & set(by_label[f"{rat} {d}"]["units"].tetrode)) for d in ["FD4", "FD9"] if f"{rat} {d}" in by_label}
    key = lambda t: int(t[2:])
    print(f"{rat}: preflight {min(pre, key=key)}..{max(pre, key=key)}, flight {min(fl, key=key)}..{max(fl, key=key)}; shared with preflight: {shared}")
    if rat == "Rat2":
        check(12, "Rat2: preflight TT0-TT10, flight TT12-TT22, none shared",
              not (pre & fl) and max(pre, key=key) == "TT10" and min(fl, key=key) == "TT12", str(shared))
    else:
        check(12, f"{rat}: 4 to 6 tetrodes shared between preflight and flight", all(4 <= v <= 6 for v in shared.values()), str(shared))

## Finding 13: data_missing

> New. The position ends before the end of two MC epochs, at 6308 s in Rat3's flight day 4 file (41 s before the epoch ends, 5.3% of its MC spikes without a position) and at 10144 s in Rat2's flight day 9 file (7 s before).

In [ ]:
# Position ends before the end of two MC epochs.
for lab, expect_end, expect_gap in [("Rat3 FD4", 6308, 41), ("Rat2 FD9", 10144, 7)]:
    s = by_label[lab]
    mc = s["epochs"][s["epochs"].type == "MC"].iloc[-1]
    own = np.concatenate(list(s["units"]["spikes"]))
    sp = own[(own >= mc.start) & (own <= mc.stop)]
    no_pos = (sp > s["t"][-1]).mean()
    print(f"{lab}: position ends at {s['t'][-1]:.0f} s; MC epoch {mc.start:.0f}-{mc.stop:.0f} s, "
          f"{mc.stop - s['t'][-1]:.0f} s before its end; MC spikes after the last position: {no_pos:.3f}")
    check(13, f"{lab}: position ends at {expect_end} s, {expect_gap} s before the MC epoch ends",
          round(s["t"][-1]) == expect_end and round(mc.stop - s["t"][-1]) == expect_gap, f"{s['t'][-1]:.0f} s, {mc.stop - s['t'][-1]:.0f} s")
    if lab == "Rat3 FD4":
        check(13, "Rat3 FD4: 5.3% of MC spikes without a position", abs(no_pos - 0.053) < 0.002, f"{no_pos:.3f}")

## Finding 14: selection_undocumented

> New. The dandiset holds more active cells than the paper counts in most comparisons (Rat3 preflight 38 active ES maps against the paper's 24; Rat3 flight day 4 session 2, 11 maps and 16 units against 7; Rat1 flight day 9, 24 against 21), and the selection behind the paper's counts is not recorded.

In [ ]:
# Active cells in the dandiset against the paper's counts. Active: a deposited map counted once whose
# occupancy-weighted mean rate exceeds 0.05 Hz (M-a), or a unit whose spikes over the epoch exceed 0.05 Hz.
# The paper's counts: 24 preflight cells of rat 3 and 7 active cells in rat 3's FD4 session 2 (Fig. 3a and
# main text); 21 active cells of rat 1 on FD9 (Fig. 2c).
ma = nl.ma_table(sessions)
def active_maps(rat, day, epoch):
    return int(ma[(ma.rat == rat) & (ma.day == day) & (ma.epoch == epoch) & ma.active].shape[0])
def active_units(rat, day, epoch):
    s = by_label[f"{rat} {day}"]
    a, b = s["epochs"].set_index("name").loc[epoch, ["start", "stop"]]
    return int(sum(((sp >= a) & (sp <= b)).sum() / (b - a) > nl.ACTIVE_HZ for sp in s["units"].spikes))
rows = [("Rat3", "preflight", "ES1", 24), ("Rat3", "FD4", "ES1", 7), ("Rat1", "FD9", "ES1", 21)]
for rat, day, ep, paper in rows:
    m, u = active_maps(rat, day, ep), active_units(rat, day, ep)
    print(f"{rat} {day} {ep}: active maps {m}, active units {u}; the paper counts {paper}")
expect = {("Rat3", "preflight"): ("maps", 38), ("Rat3", "FD4"): ("maps and units", (11, 16)), ("Rat1", "FD9"): ("maps", 24)}
check(14, "Rat3 preflight: 38 active ES maps against the paper's 24", active_maps("Rat3", "preflight", "ES1") == 38, str(active_maps("Rat3", "preflight", "ES1")))
check(14, "Rat3 FD4 session 2: 11 maps and 16 units against 7",
      (active_maps("Rat3", "FD4", "ES1"), active_units("Rat3", "FD4", "ES1")) == (11, 16),
      f"{active_maps('Rat3', 'FD4', 'ES1')} maps, {active_units('Rat3', 'FD4', 'ES1')} units")
check(14, "Rat1 FD9: 24 active against 21", 24 in (active_maps("Rat1", "FD9", "ES1"), active_units("Rat1", "FD9", "ES1")),
      f"{active_maps('Rat1', 'FD9', 'ES1')} maps, {active_units('Rat1', 'FD9', 'ES1')} units")
print("Not checked here: that nothing records the selection behind the paper's counts (an absence, read from the descriptions above).")

## Summary

Every condition checked above, and whether it holds on this version.

In [ ]:
summary = pd.DataFrame(results)
print(summary.groupby("finding").holds.agg(conditions="size", hold="sum").to_string())
print()
print(summary[~summary.holds].to_string(index=False) if (~summary.holds).any() else "Every condition holds.")